# Prometheus-1B: Phase 4 - Benchmark Evaluation
**lm-evaluation-harness를 사용한 공식 벤치마크 측정**

| 벤치마크 | 측정 항목 | 비교 대상 |
|----------|---------|----------|
| MMLU | 상식/지식 | TinyLlama, Phi-1.5, Llama 3.2 1B |
| ARC-Challenge | 과학 추론 | " |
| GSM8K | 수학 추론 | " |
| HumanEval | 코드 생성 | " |
| HellaSwag | 상식 추론 | " |
| WinoGrande | 대명사 해석 | " |
| TruthfulQA | 진실성 | " |

**예상 소요**: 2-3시간 (A100)

## 1. 환경 설정

In [ ]:
%%capture
# lm-evaluation-harness 설치 (vllm 제거 - T4 호환성 문제)
!pip install lm-eval accelerate torch transformers

In [ ]:
import os
import shutil
from google.colab import drive, userdata

drive.mount('/content/drive')
hf_token = userdata.get("HF_TOKEN")
os.environ["HF_TOKEN"] = hf_token

FINAL_MODEL_DIR  = "/content/drive/MyDrive/Prometheus-1B-Final"
RESULTS_DIR      = "/content/drive/MyDrive/Prometheus-1B-Results"
LOCAL_MODEL_DIR  = "/tmp/prometheus_model"   # SSD 경로 (빠름)
os.makedirs(RESULTS_DIR, exist_ok=True)

assert os.path.exists(FINAL_MODEL_DIR), f"Model not found at {FINAL_MODEL_DIR}. Run Phase 3 first!"

# Drive → 로컬 SSD 복사 (한 번만, 이미 있으면 스킵)
if not os.path.exists(LOCAL_MODEL_DIR):
    print("Copying Prometheus-1B from Drive to local SSD (3-5 min)...")
    shutil.copytree(FINAL_MODEL_DIR, LOCAL_MODEL_DIR)
    print("Done! Model copied to /tmp/prometheus_model")
else:
    print("Local model already exists, skipping copy.")

print(f"Results: {RESULTS_DIR}")

## 2. 벤치마크 실행
lm-evaluation-harness CLI로 주요 벤치마크를 실행합니다. 각 벤치마크를 개별적으로 실행하여 중간 실패에 강건하게 합니다.

In [ ]:
import json
import lm_eval

BENCHMARKS = [
    ("MMLU",          "mmlu",           5),
    ("ARC-Challenge",  "arc_challenge",  25),
    ("GSM8K",          "gsm8k",          5),
    ("HellaSwag",      "hellaswag",      10),
    ("WinoGrande",     "winogrande",     5),
    ("TruthfulQA",     "truthfulqa_mc2", 0),
]

EVAL_MODELS = {
    "Kybalion-1B":           LOCAL_MODEL_DIR,
    "TinyLlama-1.1B":        "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    "Llama-3.2-1B-Instruct": "meta-llama/Llama-3.2-1B-Instruct",
    # Phi-1.5 제거: PhiConfig 호환성 문제로 최신 transformers에서 동작 안 함
}

all_results = {}

for model_display_name, model_path in EVAL_MODELS.items():
    print(f"\n{'='*60}")
    print(f"Evaluating: {model_display_name}")
    print(f"{'='*60}", flush=True)

    model_results = {}

    for bench_name, task, fewshot in BENCHMARKS:
        output_file = os.path.join(
            RESULTS_DIR, f"{model_display_name.replace('/', '_')}_{task}.json"
        )

        if os.path.exists(output_file):
            print(f"  [SKIP] {bench_name} already evaluated", flush=True)
            with open(output_file) as f:
                model_results[bench_name] = json.load(f)
            continue

        print(f"  Running {bench_name} ({fewshot}-shot)...", flush=True)

        try:
            results = lm_eval.simple_evaluate(
                model="hf",
                model_args=f"pretrained={model_path},dtype=bfloat16,trust_remote_code=True",
                tasks=[task],
                num_fewshot=fewshot,
                batch_size=8,
            )
            with open(output_file, "w") as f:
                json.dump(results, f, indent=2, default=str)
            model_results[bench_name] = results
            print(f"  ✅ {bench_name} Done", flush=True)

        except Exception as e:
            print(f"  ❌ {bench_name} Error: {e}", flush=True)
            model_results[bench_name] = {"error": str(e)}

    all_results[model_display_name] = model_results

print(f"\n{'='*60}")
print("All evaluations completed!")
print(f"Results saved to: {RESULTS_DIR}")

## 3. 결과 비교 시각화
Prometheus-1B와 경쟁 모델들의 성능을 비교합니다.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

benchmark_keys = {
    "MMLU": ("mmlu", "acc,none"),
    "ARC-C": ("arc_challenge", "acc_norm,none"),
    "GSM8K": ("gsm8k", "exact_match,strict-match"),
    "HellaSwag": ("hellaswag", "acc_norm,none"),
    "WinoGrande": ("winogrande", "acc,none"),
    "TruthfulQA": ("truthfulqa_mc2", "acc,none"),
}

def extract_score(model_result_dict, task_name, metric_key):
    """lm_eval 결과에서 점수 추출"""
    try:
        results = model_result_dict.get("results", {})
        for key, vals in results.items():
            if task_name in key:
                score = vals.get(metric_key)
                if score is not None:
                    return round(score * 100, 2)
        return None
    except Exception:
        return None

# 모든 모델 점수 추출
parsed_scores = {}
for model_name, bench_results in all_results.items():
    parsed_scores[model_name] = {}
    for display_name, (task_name, metric_key) in benchmark_keys.items():
        bench_data = bench_results.get(
            "MMLU" if display_name == "MMLU" else
            "ARC-Challenge" if display_name == "ARC-C" else display_name, {}
        )
        score = extract_score(bench_data, task_name, metric_key)
        parsed_scores[model_name][display_name] = score if score is not None else 0.0

# 결과 테이블 출력
benchmarks = list(benchmark_keys.keys())
df = pd.DataFrame(parsed_scores, index=benchmarks)
print("\n" + "=" * 70)
print("Benchmark Results (directly measured, same conditions)")
print("=" * 70)
print(df.to_string())
print("=" * 70)

# 결과 JSON 저장
with open(os.path.join(RESULTS_DIR, "final_comparison.json"), "w") as f:
    json.dump(parsed_scores, f, indent=2)
print(f"\nFull results saved to: {os.path.join(RESULTS_DIR, 'final_comparison.json')}")

In [ ]:
# 바 차트
fig, ax = plt.subplots(figsize=(14, 8))

x = np.arange(len(benchmarks))
width = 0.25
models = list(parsed_scores.keys())
colors = ["#FF6B6B", "#4ECDC4", "#96CEB4"]  # 3개 모델

for i, (model_name, color) in enumerate(zip(models, colors)):
    scores = [parsed_scores[model_name].get(b, 0) for b in benchmarks]
    bars = ax.bar(x + i * width, scores, width, label=model_name, color=color, edgecolor="white")
    for bar, score in zip(bars, scores):
        if score > 0:
            ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.5,
                   f'{score:.1f}', ha='center', va='bottom', fontsize=8, fontweight='bold')

ax.set_xlabel('Benchmark', fontsize=12)
ax.set_ylabel('Score (%)', fontsize=12)
ax.set_title('Kybalion-1B vs Competitive 1B Models\n(All scores measured under identical conditions with lm-evaluation-harness)',
             fontsize=13, fontweight='bold')
ax.set_xticks(x + width)
ax.set_xticklabels(benchmarks, fontsize=11)
ax.legend(loc='upper left', fontsize=10)
ax.grid(axis='y', linestyle='--', alpha=0.5)
ax.set_ylim(0, 100)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "benchmark_comparison.png"), dpi=150, bbox_inches='tight')
plt.show()
print(f"Chart saved to: {os.path.join(RESULTS_DIR, 'benchmark_comparison.png')}")

## 4. 레이더 차트 (학술 보고서용)

In [ ]:
# 레이더 차트
fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(polar=True))

categories = benchmarks
N = len(categories)
angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]

colors_radar = ["#FF6B6B", "#4ECDC4", "#45B7D1", "#96CEB4"]

for (model_name, scores), color in zip(parsed_scores.items(), colors_radar):
    values = [scores.get(b, 0) for b in categories]
    values += values[:1]
    ax.plot(angles, values, 'o-', linewidth=2, label=model_name, color=color)
    ax.fill(angles, values, alpha=0.1, color=color)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=11)
ax.set_ylim(0, 80)
ax.set_title("Kybalion-1B: Multi-Benchmark Radar", fontsize=14, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=10)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "radar_chart.png"), dpi=150, bbox_inches='tight')
plt.show()

print(f"\nRadar chart saved to: {os.path.join(RESULTS_DIR, 'radar_chart.png')}")
print(f"\nEvaluation complete! Proceed to Phase 5 (Deploy) with notebook 05_deploy.ipynb")